# Lab 02 — Run an LLM and see the machinery

**Goal:** load a real transformer, watch text → tokens → logits → probabilities → next token, count parameters against the 12H² rule, and inspect the anatomy (attention + MLP matrices) from the proposal's background.

Model: **GPT-2 (124M)** — small, fast, structurally identical to Llama-class models.

In [ ]:
!pip -q install transformers
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
print("loaded on", device)

## 1. Tokens are just integers

In [ ]:
text = "Pipeline parallelism splits a model across"
ids = tok(text, return_tensors="pt").to(device)
print(ids.input_ids)
print([tok.decode(t) for t in ids.input_ids[0]])
print("vocab size:", tok.vocab_size)

## 2. One forward pass = one probability distribution per position
`logits` has shape `[B, S, V]` — a score for every vocabulary word at every position. Softmax on the **last** position gives the next-token distribution.

In [ ]:
with torch.no_grad():
    out = model(**ids)
print("logits:", out.logits.shape)
probs = out.logits[0, -1].softmax(-1)
topk = probs.topk(5)
for p, t in zip(topk.values, topk.indices):
    print(f"{p.item():.3f}  {tok.decode(t)!r}")

## 3. Generation = the forward pass in a loop
Greedy decoding by hand, then the library version.

In [ ]:
cur = ids.input_ids
with torch.no_grad():
    for _ in range(12):
        nxt = model(cur).logits[0, -1].argmax()
        cur = torch.cat([cur, nxt.view(1,1)], dim=1)
print("manual :", tok.decode(cur[0]))
gen = model.generate(**ids, max_new_tokens=12, do_sample=False)
print("library:", tok.decode(gen[0]))

## 4. Parameter accounting — check the 12H² rule
Per block ≈ 4H² (attention) + 8H² (MLP, 4x expansion) = 12H². GPT-2: H=768, L=12, plus a V×H embedding.

In [ ]:
H, L, V = 768, 12, 50257
predicted = 12*H*H*L + V*H
actual = sum(p.numel() for p in model.parameters())
print(f"predicted ~{predicted/1e6:.0f}M   actual {actual/1e6:.0f}M")

Close enough — the difference is layer norms, biases, and position embeddings. You can now estimate any transformer's size from H and L on a whiteboard.

## 5. Anatomy — where the matrices live

In [ ]:
blk = model.transformer.h[0]
print(blk)
print()
print("attn c_attn weight:", blk.attn.c_attn.weight.shape, " (fused W_Q,W_K,W_V -> 3H wide)")
print("mlp  c_fc   weight:", blk.mlp.c_fc.weight.shape,  " (up-projection H -> 4H)")
print("mlp  c_proj weight:", blk.mlp.c_proj.weight.shape, " (down-projection 4H -> H)")

## Exercises
1. Sample with temperature: divide logits by `T` before softmax, use `torch.multinomial`. Compare T=0.7 vs T=1.5 outputs.
2. Compute the model's average cross-entropy on a paragraph using `model(input_ids, labels=input_ids).loss` — you'll reuse this as the quality metric in Lab 04.
3. Load `Qwen/Qwen2.5-0.5B` and redo the parameter accounting (note: it uses grouped-query attention, so attention is a bit under 4H²).